# Travel Agents

Starts the three ADK-backed travel agents: Air Ticketing, Hotel Booking, and Car Rental.
Each connects to the MCP server for tool access (SQL queries against `travel_agency.db`).

In [1]:
import asyncio
import json
import logging
import os
import re
import sys
import uuid
from collections.abc import AsyncIterable
from typing import Any

import litellm
import uvicorn
import weave
from dotenv import load_dotenv
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.mcp_tool.mcp_session_manager import SseServerParams
from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset
from google.genai import types as genai_types

import prompts
from common import MCP_HOST, MCP_PORT, BaseAgent, build_a2a_app

load_dotenv()
logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s',
    stream=sys.stdout,
    force=True,
)
logger.setLevel(logging.INFO)

WANDB_PROJECT = os.getenv('WANDB_WORKSPACE')
weave_client = weave.init(WANDB_PROJECT)

# Explicitly patch litellm so Weave traces all LLM calls made by ADK agents
from weave.integrations.patch import patch_litellm
patch_litellm()

print(f'Weave initialized: {WANDB_PROJECT}')

/Users/paul/.cache/uv/archive-v0/pqxv7ELpHFQOMM0CmHnVk/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(
/Users/paul/Documents/code/a2a-examples/.venv/lib/python3.13/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils
weave: weave version 0.52.37 is available!  To upgrade, please run:
weave:  $ pip install weave --upgrade


2026-04-20 13:12:20,879 INFO weave.trace.init_message: weave version 0.52.37 is available!  To upgrade, please run:
 $ pip install weave --upgrade


weave: Logged in as Weights & Biases user: paulbruffett.
weave: View Weave data at https://wandb.ai/pbruffett/a2a-lab/weave


2026-04-20 13:12:20,880 INFO weave.trace.init_message: Logged in as Weights & Biases user: paulbruffett.
View Weave data at https://wandb.ai/pbruffett/a2a-lab/weave
Weave initialized: pbruffett/a2a-lab


In [2]:
class TravelAgent(BaseAgent):
    """ADK-backed travel agent that connects to the MCP server for tools."""

    def __init__(self, agent_name: str, description: str, instructions: str):
        super().__init__(agent_name=agent_name, description=description, content_types=['text', 'text/plain'])
        self.instructions = instructions
        self.agent = None
        self.session_service = InMemorySessionService()
        self.app_name = 'A2A-MCP'
        self.user_id = 'user_1'

    async def _init_agent(self):
        url = f'http://{MCP_HOST}:{MCP_PORT}/sse'
        tools = await MCPToolset(connection_params=SseServerParams(url=url)).get_tools()
        model = 'anthropic/claude-haiku-4-5'
        self.agent = Agent(
            name=self.agent_name, instruction=self.instructions,
            model=LiteLlm(model=model),
            disallow_transfer_to_parent=True, disallow_transfer_to_peers=True,
            generate_content_config=genai_types.GenerateContentConfig(temperature=0.0),
            tools=tools,
        )

    @weave.op()
    async def stream(self, query, context_id, task_id) -> AsyncIterable[dict[str, Any]]:
        logger.info('[%s.stream] REQUEST ctx=%s task=%s query=%r', self.agent_name, context_id, task_id, query)
        if not query:
            raise ValueError('Query cannot be empty')
        if not self.agent:
            await self._init_agent()

        session_id = context_id or uuid.uuid4().hex
        session = await self.session_service.get_session(
            app_name=self.app_name, user_id=self.user_id, session_id=session_id,
        )
        if not session:
            session = await self.session_service.create_session(
                app_name=self.app_name, user_id=self.user_id, session_id=session_id,
            )

        runner = Runner(agent=self.agent, app_name=self.app_name, session_service=self.session_service)
        content = genai_types.Content(role='user', parts=[genai_types.Part(text=query)])
        async for event in runner.run_async(user_id=self.user_id, session_id=session.id, new_message=content):
            if event.is_final_response():
                if event.content and event.content.parts and event.content.parts[0].text:
                    response = '\n'.join(p.text for p in event.content.parts if p.text)
                elif event.content and event.content.parts and any(p.function_response for p in event.content.parts):
                    response = next(p.function_response.model_dump() for p in event.content.parts)
                else:
                    response = f'Error in running agent: {self.agent.name}'
                parsed = self._parse_response(response)
                logger.info('[%s.stream] FINAL %s', self.agent_name, parsed)
                yield parsed
            else:
                yield {'is_task_complete': False, 'require_user_input': False, 'content': f'{self.agent_name}: Processing...'}

    @weave.op()
    def _parse_response(self, raw):
        data = raw
        for pattern in [r'```\n(.*?)\n```', r'```json\s*(.*?)\s*```', r'```tool_outputs\s*(.*?)\s*```']:
            match = re.search(pattern, str(raw), re.DOTALL)
            if match:
                try:
                    data = json.loads(match.group(1))
                    break
                except json.JSONDecodeError:
                    data = match.group(1)
                    break
        if isinstance(data, dict):
            if data.get('status') == 'input_required':
                return {'response_type': 'text', 'is_task_complete': False, 'require_user_input': True, 'content': data['question']}
            return {'response_type': 'data', 'is_task_complete': True, 'require_user_input': False, 'content': data}
        try:
            data = json.loads(data)
            return {'response_type': 'data', 'is_task_complete': True, 'require_user_input': False, 'content': data}
        except Exception:
            return {'response_type': 'text', 'is_task_complete': True, 'require_user_input': False, 'content': data}

In [ ]:
agents = [
    (TravelAgent('AirTicketingAgent', 'Book air tickets', prompts.AIRFARE_COT_INSTRUCTIONS), 'agent_cards/air_ticketing_agent.json', 10103),
    (TravelAgent('HotelBookingAgent', 'Book hotels', prompts.HOTELS_COT_INSTRUCTIONS), 'agent_cards/hotel_booking_agent.json', 10104),
    (TravelAgent('CarRentalBookingAgent', 'Book rental cars', prompts.CARS_COT_INSTRUCTIONS), 'agent_cards/car_rental_agent.json', 10105),
]

async def serve_all():
    servers = []
    for agent, card_path, port in agents:
        app = build_a2a_app(agent, card_path)
        config = uvicorn.Config(app=app, host='localhost', port=port, log_level='info',)
        server = uvicorn.Server(config)
        servers.append(server)
        print(f'{agent.agent_name} running at http://localhost:{port}')
    await asyncio.gather(*(s.serve() for s in servers))

await serve_all()

AirTicketingAgent running at http://localhost:10103
HotelBookingAgent running at http://localhost:10104


INFO:     Started server process [28858]
INFO:     Waiting for application startup.
INFO:     Started server process [28858]
INFO:     Waiting for application startup.
INFO:     Started server process [28858]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Application startup complete.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:10103 (Press CTRL+C to quit)
INFO:     Uvicorn running on http://localhost:10105 (Press CTRL+C to quit)
INFO:     Uvicorn running on http://localhost:10104 (Press CTRL+C to quit)


CarRentalBookingAgent running at http://localhost:10105
INFO:     ::1:61430 - "POST / HTTP/1.1" 200 OK
2026-04-20 13:13:33,317 INFO __main__: [AirTicketingAgent.stream] REQUEST ctx=4fc6677d-6b1b-4867-87ed-764b1cc27ed8 task=aac41822-d3fd-4696-a1b5-6e17cb39255c query='Book round-trip premium economy class air tickets from San Francisco (SFO) to London (LHR) for 2 travelers, departing May 1, 2026 and returning May 11, 2026.'


/var/folders/ch/wjmp4fss3gvbg92jslrpt8_m0000gn/T/ipykernel_28858/2071780325.py:14: DeprecationWarning: MCPToolset class is deprecated, use `McpToolset` instead.
  tools = await MCPToolset(connection_params=SseServerParams(url=url)).get_tools()
weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019dac86-df05-7266-89a9-1a4b4b5cdbc6


2026-04-20 13:13:33,320 INFO weave.trace.weave_client: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019dac86-df05-7266-89a9-1a4b4b5cdbc6
2026-04-20 13:13:33,347 WARNING google_adk.google.adk.tools.base_authenticated_tool: auth_config or auth_config.auth_scheme is missing. Will skip authentication.Using FunctionTool instead if authentication is not required.
2026-04-20 13:13:33,347 WARNING google_adk.google.adk.tools.base_authenticated_tool: auth_config or auth_config.auth_scheme is missing. Will skip authentication.Using FunctionTool instead if authentication is not required.
2026-04-20 13:13:33,347 WARNING google_adk.google.adk.tools.base_authenticated_tool: auth_config or auth_config.auth_scheme is missing. Will skip authentication.Using FunctionTool instead if authentication is not required.
2026-04-20 13:13:33,349 INFO google_genai.types: 
Note: Conversion of fields that are not included in the JSONSchema class are ignored.
Json Schema is now supported natively by both Vertex AI and

...

2026-04-20 13:13:52,723 INFO LiteLLM: 
LiteLLM completion() model= claude-haiku-4-5; provider = anthropic
2026-04-20 13:13:55,260 INFO __main__: [HotelBookingAgent.stream] FINAL {'response_type': 'data', 'is_task_complete': True, 'require_user_input': False, 'content': {'name': 'Luxury Hotel', 'city': 'London', 'hotel_type': 'HOTEL', 'room_type': 'SUITE', 'price_per_night': '$189.46', 'check_in_date': 'May 1, 2026', 'check_in_time': '3:00 pm', 'check_out_date': 'May 11, 2026', 'check_out_time': '11:00 am', 'number_of_nights': 10, 'number_of_guests': 2, 'total_rate_usd': '$1,894.60', 'status': 'BOOKING_CONFIRMED', 'description': 'Booking Complete'}}
INFO:     ::1:61452 - "POST / HTTP/1.1" 200 OK
2026-04-20 13:13:56,190 INFO __main__: [CarRentalBookingAgent.stream] REQUEST ctx=652c87a5-2765-4814-8cb9-990c0033f980 task=f8e94511-cab0-49b8-9374-a0ea4f860a7b query='No car rental required for this trip.'


weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019dac87-385e-7f22-8de0-3dbad3148654


2026-04-20 13:13:56,191 INFO weave.trace.weave_client: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019dac87-385e-7f22-8de0-3dbad3148654
2026-04-20 13:13:56,221 WARNING google_adk.google.adk.tools.base_authenticated_tool: auth_config or auth_config.auth_scheme is missing. Will skip authentication.Using FunctionTool instead if authentication is not required.
2026-04-20 13:13:56,221 WARNING google_adk.google.adk.tools.base_authenticated_tool: auth_config or auth_config.auth_scheme is missing. Will skip authentication.Using FunctionTool instead if authentication is not required.
2026-04-20 13:13:56,222 WARNING google_adk.google.adk.tools.base_authenticated_tool: auth_config or auth_config.auth_scheme is missing. Will skip authentication.Using FunctionTool instead if authentication is not required.


13:13:56 - LiteLLM:INFO: utils.py:4004 - 
LiteLLM completion() model= claude-haiku-4-5; provider = anthropic


2026-04-20 13:13:56,223 INFO LiteLLM: 
LiteLLM completion() model= claude-haiku-4-5; provider = anthropic
2026-04-20 13:13:57,399 INFO __main__: [CarRentalBookingAgent.stream] FINAL {'response_type': 'text', 'is_task_complete': True, 'require_user_input': False, 'content': "I understand! You won't need a car rental for this trip. \n\nIf you change your mind or need a car rental in the future, feel free to reach out and I'll be happy to help you book one.\n\nIs there anything else I can assist you with?"}
